In [0]:
-- Create Quantexa Data Product organisations_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata ( 
	supplier STRING,
	data_product STRING,
	data_product_version STRING,
	schema_name STRING,
	schema_version STRING,
	schema_variant_name STRING,
	schema_variant_version STRING,
	instance STRING,
	instance_version STRING
 );

INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.organisation_suppliers',
  '0.1',
  'organisation_suppliers.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);


In [0]:
-- Create Quantexa Data Product organisation_suppliers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Organisation Customer table
DROP TABLE IF EXISTS hub_organisation_suppliers;

CREATE OR REPLACE TABLE hub_organisation_suppliers (
    organisation_supplier_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Organisation Supplier Identity',
    organisation_supplier_entity_id STRING,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
    address_entity_id STRING,
    organisation_entity_id STRING,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE hub_organisation_suppliers ADD CONSTRAINT pk_organisationsuppliers_huborganisationsuppliers_organisationsupplierid PRIMARY KEY (organisation_supplier_id);


In [0]:
-- Create Quantexa Data Product organisation_suppliers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Organisation Customer table
DROP TABLE IF EXISTS sat_organisation_suppliers;

CREATE OR REPLACE TABLE sat_organisation_suppliers (
    sat_organisation_supplier_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Organisation Supplier Identity Sat',
    organisation_supplier_id BIGINT NOT NULL COMMENT 'Organisation Supplier Identity',
    organisation_id BIGINT COMMENT 'Organisation Identity',
    address_id BIGINT COMMENT 'Address Identity',
		supplier_name STRING,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    record_source_id BIGINT NOT NULL DEFAULT 1,
	  data_source_code STRING,
	  data_source_concept_id BIGINT,
	  data_source_raw_code STRING,
	  data_source_raw_concept_id BIGINT,
	  type_code STRING,
	  type_concept_id BIGINT,
	  type_raw_code STRING,
	  type_raw_concept_id BIGINT,
	  line_of_business_code STRING,
	  line_of_business_concept_id BIGINT,
	  line_of_business_raw_code STRING,
	  line_of_business_raw_concept_id BIGINT,
	  importance_code STRING,
	  importance_concept_id BIGINT,
	  importance_raw_code STRING,
	  importance_raw_concept_id BIGINT,
	  status_code STRING,
	  status_concept_id BIGINT,
	  status_raw_code STRING,
	  status_raw_concept_id BIGINT,
		risk_rating INT,
		risk_rating_raw STRING,
		risk_rating_code STRING,
		risk_rating_concept_id BIGINT,
		risk_rating_raw_code STRING,
		risk_rating_raw_concept_id BIGINT,
		is_supplier_alarm BOOLEAN DEFAULT false,
		is_offboarded BOOLEAN DEFAULT false,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_organisation_suppliers ADD CONSTRAINT pk_organisationsuppliers_satorganisationsuppliers_satorganisationsupplierid PRIMARY KEY (sat_organisation_supplier_id);
ALTER TABLE sat_organisation_suppliers ADD CONSTRAINT fk_organisationsuppliers_satorganisationsuppliers_huborganisationsuppliers_organisationsupplierid FOREIGN KEY (organisation_supplier_id) REFERENCES hub_organisation_suppliers(organisation_supplier_id);


In [0]:
-- Create Quantexa Data Product organisation_suppliers Schema

-- Organisation Supplier Risk Score table
DROP TABLE IF EXISTS sat_organisation_supplier_risk_scores;

CREATE OR REPLACE TABLE sat_organisation_supplier_risk_scores (
	organisation_supplier_risk_score_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Organisation Supplier Risk Score Identity',
	organisation_supplier_id BIGINT NOT NULL,
  load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  record_source_id BIGINT NOT NULL DEFAULT 1,
	risk_code STRING,
	risk_concept_id BIGINT,
	risk_raw_code STRING,
	risk_raw_concept_id BIGINT,
	risk_score STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_organisation_supplier_risk_scores ADD CONSTRAINT pk_organisationsuppliers_organisationsupplierriskscores_organisationsupplierriskscoreid PRIMARY KEY (organisation_supplier_risk_score_id);
ALTER TABLE sat_organisation_supplier_risk_scores ADD CONSTRAINT fk_organisationsuppliers_organisationsupplierriskscores_huborganisationsuppliers_organisationsupplierid FOREIGN KEY (organisation_supplier_id) REFERENCES hub_organisation_suppliers(organisation_supplier_id);



In [0]:
-- Create Quantexa Data Product organisation_suppliers Schema

-- Organisation Suppliers Warning Signals table
DROP TABLE IF EXISTS sat_organisation_supplier_warning_signals;

CREATE OR REPLACE TABLE sat_organisation_supplier_warning_signals (
	organisation_supplier_warning_signals_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Organisation Supplier Warning Signals Identity',
	organisation_supplier_id BIGINT NOT NULL,
  load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  record_source_id BIGINT NOT NULL DEFAULT 1,
	warning_code STRING,
	warning_concept_id BIGINT,
	warning_raw_code STRING,
	warning_raw_concept_id BIGINT,
	warning_signal STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_organisation_supplier_warning_signals ADD CONSTRAINT pk_organisationsupplier_organisationsupplierwarningsignals_organisationsupplierwarningsignalsid PRIMARY KEY (organisation_supplier_warning_signals_id);
ALTER TABLE sat_organisation_supplier_warning_signals ADD CONSTRAINT fk_organisationsuppliers_organisationsupplierwarningsignals_huborganisationsuppliers_organisationsupplierid FOREIGN KEY (organisation_supplier_id) REFERENCES hub_organisation_suppliers(organisation_supplier_id);


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_suppliers AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.organisation_supplier_id,
  s.supplier_name,
  h.organisation_supplier_entity_id, 
  h.organisation_entity_id, 
  h.period_start as supplier_period_start,
  h.period_end as supplier_period_end,
  s.type_code as supplier_type_code,
  s.status_code as supplier_status_code,
  s.risk_rating,
  s.is_supplier_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  horg.organisation_id, 
	horg.period_start AS hub_period_start,
	horg.period_end AS hub_period_end,
  sorg.sat_organisation_id,
  sorg.type_code as organisation_type_code,
  sorg.type_concept_id AS organisation_type_concept_id,
  sorg.type_raw_code AS organisation_type_raw_code,
  sorg.type_raw_concept_id AS organisation_type_raw_concept_id,
  sorg.organisation_name,
  sorg.organisation_name_raw,
  sorg.organisation_name_parsed,
  sorg.trading_name,
  sorg.trading_name_raw,
  sorg.alternative_name,
  sorg.alternative_name_raw,
  sorg.duns_number,
  sorg.acronym,
  sorg.organisation_description,
  sorg.organisation_level,
  sorg.parent_organisation_id,
  sorg.primary_adddress_id,
  sorg.legal_structure_type_code,
  sorg.legal_structure_type_concept_id,
  sorg.legal_structure_type_raw_code,
  sorg.legal_structure_type_raw_concept_id,
  sorg.ring_fence_status,
  sorg.company_category_code,
  sorg.company_category_concept_id,
  sorg.company_category_raw_code,
  sorg.company_category_raw_concept_id,
  sorg.company_registration_name,
  sorg.company_registration_number,
  sorg.company_registration_date,
  sorg.company_registration_date_numeric,
  sorg.company_registration_place,
  sorg.company_registration_country_code,
  sorg.company_registration_country_concept_id,
  sorg.company_registration_country_raw_code,
  sorg.company_registration_country_raw_concept_id,
  sorg.primary_operation_country_code,
  sorg.primary_operation_country_concept_id,
  sorg.primary_operation_country_raw_code,
  sorg.primary_operation_country_raw_concept_id,
  sorg.secondary_operation_country_code,
  sorg.secondary_operation_country_concept_id,
  sorg.secondary_operation_country_raw_code,
  sorg.secondary_operation_country_raw_concept_id,
  sorg.company_franchise_type_code,
  sorg.company_franchise_type_concept_id,
  sorg.company_franchise_type_raw_code,
  sorg.company_franchise_type_raw_concept_id,
  sorg.vat_number,
  sorg.vat_registration_date,
  sorg.vat_registration_date_numeric,
  sorg.giin,
  sorg.bic,
  sorg.iban,
  sorg.legal_entity_identifier,
  sorg.firm_reference_number,
  sorg.unique_tax_payer_reference,
  sorg.capiq,
  sorg.risk_rating_concept_id,
  sorg.risk_rating_raw_code,
  sorg.risk_rating_raw_concept_id,
  sorg.period_start,
	sorg.period_end,
	sorg.load_datetime,
	sorg.record_source_id,
	sorg.data_source_code,
	sorg.data_source_concept_id,
	sorg.data_source_raw_code,
	sorg.data_source_raw_concept_id


    FROM demo_banking_silver.qdp_organisation_suppliers.hub_organisation_suppliers h
    JOIN demo_banking_silver.qdp_organisation_suppliers.sat_organisation_suppliers s
      ON h.organisation_supplier_id = s.organisation_supplier_id
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
      ON horg.organisation_id = s.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation sorg
      ON horg.organisation_id = sorg.organisation_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
    ORDER BY h.organisation_supplier_id
;

SELECT * from view_organisation_suppliers;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_supplier_addresses AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.organisation_supplier_id,
  s.supplier_name,
  sadd.full_address as supplier_full_address,
  sadd.postal_code as supplier_full_address_postal_code,
  h.organisation_supplier_entity_id, 
  h.organisation_entity_id, 
  h.period_start as supplier_period_start,
  h.period_end as supplier_period_end,
  s.type_code as supplier_type_code,
  s.status_code as supplier_status_code,
  s.risk_rating,
  s.is_supplier_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  horg.organisation_id, 
	horg.period_start AS hub_period_start,
	horg.period_end AS hub_period_end,
  sorg.sat_organisation_id,
  sorg.type_code as organisation_type_code,
  sorg.type_concept_id AS organisation_type_concept_id,
  sorg.type_raw_code AS organisation_type_raw_code,
  sorg.type_raw_concept_id AS organisation_type_raw_concept_id,
  sorg.organisation_name,
  sorg.organisation_name_raw,
  sorg.organisation_name_parsed,
  sorg.trading_name,
  sorg.trading_name_raw,
  sorg.alternative_name,
  sorg.alternative_name_raw,
  sorg.duns_number,
  sorg.acronym,
  sorg.organisation_description,
  sorg.organisation_level,
  sorg.parent_organisation_id,
  sorg.primary_adddress_id,
  sorg.legal_structure_type_code,
  sorg.legal_structure_type_concept_id,
  sorg.legal_structure_type_raw_code,
  sorg.legal_structure_type_raw_concept_id,
  sorg.ring_fence_status,
  sorg.company_category_code,
  sorg.company_category_concept_id,
  sorg.company_category_raw_code,
  sorg.company_category_raw_concept_id,
  sorg.company_registration_name,
  sorg.company_registration_number,
  sorg.company_registration_date,
  sorg.company_registration_date_numeric,
  sorg.company_registration_place,
  sorg.company_registration_country_code,
  sorg.company_registration_country_concept_id,
  sorg.company_registration_country_raw_code,
  sorg.company_registration_country_raw_concept_id,
  sorg.primary_operation_country_code,
  sorg.primary_operation_country_concept_id,
  sorg.primary_operation_country_raw_code,
  sorg.primary_operation_country_raw_concept_id,
  sorg.secondary_operation_country_code,
  sorg.secondary_operation_country_concept_id,
  sorg.secondary_operation_country_raw_code,
  sorg.secondary_operation_country_raw_concept_id,
  sorg.company_franchise_type_code,
  sorg.company_franchise_type_concept_id,
  sorg.company_franchise_type_raw_code,
  sorg.company_franchise_type_raw_concept_id,
  sorg.vat_number,
  sorg.vat_registration_date,
  sorg.vat_registration_date_numeric,
  sorg.giin,
  sorg.bic,
  sorg.iban,
  sorg.legal_entity_identifier,
  sorg.firm_reference_number,
  sorg.unique_tax_payer_reference,
  sorg.capiq,
  sorg.risk_rating_concept_id,
  sorg.risk_rating_raw_code,
  sorg.risk_rating_raw_concept_id,
  sorg.period_start,
	sorg.period_end,
	sorg.load_datetime,
	sorg.record_source_id,
	sorg.data_source_code,
	sorg.data_source_concept_id,
	sorg.data_source_raw_code,
	sorg.data_source_raw_concept_id


    FROM demo_banking_silver.qdp_organisation_suppliers.hub_organisation_suppliers h
    JOIN demo_banking_silver.qdp_organisation_suppliers.sat_organisation_suppliers s
      ON h.organisation_supplier_id = s.organisation_supplier_id
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
      ON horg.organisation_id = s.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation sorg
      ON horg.organisation_id = sorg.organisation_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address hadd
      ON s.address_id = hadd.address_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address sadd
      ON hadd.address_id = sadd.address_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
    ORDER BY h.organisation_supplier_id ASC
;

SELECT * from view_organisation_supplier_addresses;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_supplier_contact_details AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.organisation_supplier_id,
  s.supplier_name,
  ocm.rank AS communication_method_rank,
  ocm.is_primary_communication_method,
  ocm.value AS contact_value,
  ocm.type_code AS contact_type_code,
  ocm.use_code AS contact_use_code,
  ocm.telephony_country_code,
  ocm.period_start AS contact_method_start,
  ocm.period_end AS contact_method_end,
  sadd.full_address,
  sadd.postal_code,
  s.address_id,
  h.organisation_supplier_entity_id, 
  h.organisation_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating,
  s.is_supplier_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  horg.organisation_id, 
	horg.period_start AS hub_period_start,
	horg.period_end AS hub_period_end,
  sorg.sat_organisation_id,
  sorg.type_code as organisation_type_code,
  sorg.type_concept_id AS organisation_type_concept_id,
  sorg.type_raw_code AS organisation_type_raw_code,
  sorg.type_raw_concept_id AS organisation_type_raw_concept_id,
  sorg.organisation_name,
  sorg.organisation_name_raw,
  sorg.organisation_name_parsed,
  sorg.trading_name,
  sorg.trading_name_raw,
  sorg.alternative_name,
  sorg.alternative_name_raw,
  sorg.duns_number,
  sorg.acronym,
  sorg.organisation_description,
  sorg.organisation_level,
  sorg.parent_organisation_id,
  sorg.primary_adddress_id,
  sorg.legal_structure_type_code,
  sorg.legal_structure_type_concept_id,
  sorg.legal_structure_type_raw_code,
  sorg.legal_structure_type_raw_concept_id,
  sorg.ring_fence_status,
  sorg.company_category_code,
  sorg.company_category_concept_id,
  sorg.company_category_raw_code,
  sorg.company_category_raw_concept_id,
  sorg.company_registration_name,
  sorg.company_registration_number,
  sorg.company_registration_date,
  sorg.company_registration_date_numeric,
  sorg.company_registration_place,
  sorg.company_registration_country_code,
  sorg.company_registration_country_concept_id,
  sorg.company_registration_country_raw_code,
  sorg.company_registration_country_raw_concept_id,
  sorg.primary_operation_country_code,
  sorg.primary_operation_country_concept_id,
  sorg.primary_operation_country_raw_code,
  sorg.primary_operation_country_raw_concept_id,
  sorg.secondary_operation_country_code,
  sorg.secondary_operation_country_concept_id,
  sorg.secondary_operation_country_raw_code,
  sorg.secondary_operation_country_raw_concept_id,
  sorg.company_franchise_type_code,
  sorg.company_franchise_type_concept_id,
  sorg.company_franchise_type_raw_code,
  sorg.company_franchise_type_raw_concept_id,
  sorg.vat_number,
  sorg.vat_registration_date,
  sorg.vat_registration_date_numeric,
  sorg.giin,
  sorg.bic,
  sorg.iban,
  sorg.legal_entity_identifier,
  sorg.firm_reference_number,
  sorg.unique_tax_payer_reference,
  sorg.capiq,
  sorg.risk_rating_concept_id,
  sorg.risk_rating_raw_code,
  sorg.risk_rating_raw_concept_id,
  sorg.period_start,
	sorg.period_end,
	sorg.load_datetime,
	sorg.record_source_id,
	sorg.data_source_code,
	sorg.data_source_concept_id,
	sorg.data_source_raw_code,
	sorg.data_source_raw_concept_id

    FROM demo_banking_silver.qdp_organisation_suppliers.hub_organisation_suppliers h
    JOIN demo_banking_silver.qdp_organisation_suppliers.sat_organisation_suppliers s
      ON h.organisation_supplier_id = s.organisation_supplier_id
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
      ON horg.organisation_id = s.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation sorg
      ON horg.organisation_id = sorg.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_communication_method ocm 
      ON horg.organisation_id = ocm.organisation_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address hadd
      ON s.address_id = hadd.address_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address sadd
      ON hadd.address_id = sadd.address_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
    ORDER BY h.organisation_supplier_id ASC
;

SELECT * from view_organisation_supplier_contact_details;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_supplier_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.organisation_supplier_id,
  s.supplier_name,
	hacc.period_start AS account_opened,
	hacc.period_end AS account_closed,
  sacc.sort_code,
  sacc.account_number,
  sacc.account_name,
  sacc.iban,
  sacc.swiftbic,
  saddr.full_address AS account_address,
  saddr.postal_code AS account_postal_code,
  sacc.type_code,
  sacc.balance,
  sacc.overdraft_limit,
  sacc.loan_original_amount,
  sacc.loan_orininal_maturity_date,
  sacc.servicer_identification,
  sacc.servicer_scheme_name,    
  sacc.currency_code,
  sacc.branch_code,
  sacc.opened_datetime,
  sacc.closed_datetime,
  sacc.status_code,
  sacc.risk_rating as account_risk_rating,
  sacc.risk_rating_code as account_risk_rating_code,
  sacc.country_code,sacc.is_account_alarm,
  sacc.is_business_account,
  sacc.is_customer_acccount,
  sacc.is_counterparty_account,
  sacc.is_frozed,
  sacc.is_closed,
  sacc.is_open,
  sacc.is_excluded_from_monitoring,
  h.organisation_supplier_entity_id, 
  h.organisation_entity_id, 
  h.period_start as supplier_period_start,
  h.period_end as supplier_period_end,
  s.type_code as supplier_type_code,
  s.status_code as supplier_status_code,
  s.risk_rating as supplier_risk_rating,
  s.is_supplier_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code as customer_risk_rting_code,
  horg.organisation_id, 
	horg.period_start AS hub_period_start,
	horg.period_end AS hub_period_end,
  sorg.sat_organisation_id,
  sorg.type_code as organisation_type_code,
  sorg.type_concept_id AS organisation_type_concept_id,
  sorg.type_raw_code AS organisation_type_raw_code,
  sorg.type_raw_concept_id AS organisation_type_raw_concept_id,
  sorg.organisation_name,
  sorg.organisation_name_raw,
  sorg.organisation_name_parsed,
  sorg.trading_name,
  sorg.trading_name_raw,
  sorg.alternative_name,
  sorg.alternative_name_raw,
  sorg.duns_number,
  sorg.acronym,
  sorg.organisation_description,
  sorg.organisation_level,
  sorg.parent_organisation_id,
  sorg.primary_adddress_id,
  sorg.legal_structure_type_code,
  sorg.legal_structure_type_concept_id,
  sorg.legal_structure_type_raw_code,
  sorg.legal_structure_type_raw_concept_id,
  sorg.ring_fence_status,
  sorg.company_category_code,
  sorg.company_category_concept_id,
  sorg.company_category_raw_code,
  sorg.company_category_raw_concept_id,
  sorg.company_registration_name,
  sorg.company_registration_number,
  sorg.company_registration_date,
  sorg.company_registration_date_numeric,
  sorg.company_registration_place,
  sorg.company_registration_country_code,
  sorg.company_registration_country_concept_id,
  sorg.company_registration_country_raw_code,
  sorg.company_registration_country_raw_concept_id,
  sorg.primary_operation_country_code,
  sorg.primary_operation_country_concept_id,
  sorg.primary_operation_country_raw_code,
  sorg.primary_operation_country_raw_concept_id,
  sorg.secondary_operation_country_code,
  sorg.secondary_operation_country_concept_id,
  sorg.secondary_operation_country_raw_code,
  sorg.secondary_operation_country_raw_concept_id,
  sorg.company_franchise_type_code,
  sorg.company_franchise_type_concept_id,
  sorg.company_franchise_type_raw_code,
  sorg.company_franchise_type_raw_concept_id,
  sorg.vat_number,
  sorg.vat_registration_date,
  sorg.vat_registration_date_numeric,
  sorg.giin,
  sorg.bic AS organisation_bic,
  sorg.iban AS organisation_iban,
  sorg.legal_entity_identifier,
  sorg.firm_reference_number,
  sorg.unique_tax_payer_reference,
  sorg.capiq,
  sorg.risk_rating_concept_id,
  sorg.risk_rating_raw_code,
  sorg.risk_rating_raw_concept_id,
  sorg.period_start,
	sorg.period_end,
	sorg.load_datetime,
	sorg.record_source_id,
	sorg.data_source_code,
	sorg.data_source_concept_id,
	sorg.data_source_raw_code,
	sorg.data_source_raw_concept_id

    FROM demo_banking_silver.qdp_organisation_suppliers.hub_organisation_suppliers h
    JOIN demo_banking_silver.qdp_organisation_suppliers.sat_organisation_suppliers s
      ON h.organisation_supplier_id = s.organisation_supplier_id
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
      ON horg.organisation_id = s.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation sorg
      ON horg.organisation_id = sorg.organisation_id
    JOIN demo_banking_silver.qdp_accounts_all.sat_account sacc 
      ON h.organisation_supplier_id = sacc.organisation_customer_id
    JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc 
      ON sacc.account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address saddr
      ON sacc.address_id = saddr.address_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
      ON saddr.address_id = haddr.address_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
    ORDER BY organisation_supplier_id
;

SELECT * from view_organisation_supplier_accounts;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_organisation_supplier_bank_accounts_with_alarm AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.organisation_supplier_id,
  s.supplier_name,
	hacc.period_start AS account_opened,
	hacc.period_end AS account_closed,
  sacc.sort_code,
  sacc.account_number,
  sacc.account_name,
  sacc.iban,
  sacc.swiftbic,
  saddr.full_address AS account_address,
  saddr.postal_code AS account_postal_code,
  sacc.type_code,
  sacc.balance,
  sacc.overdraft_limit,
  sacc.loan_original_amount,
  sacc.loan_orininal_maturity_date,
  sacc.servicer_identification,
  sacc.servicer_scheme_name,    
  sacc.currency_code,
  sacc.branch_code,
  sacc.opened_datetime,
  sacc.closed_datetime,
  sacc.status_code,
  sacc.risk_rating as account_risk_rating,
  sacc.risk_rating_code as account_risk_rating_code,
  sacc.country_code,sacc.is_account_alarm,
  sacc.is_business_account,
  sacc.is_customer_acccount,
  sacc.is_counterparty_account,
  sacc.is_frozed,
  sacc.is_closed,
  sacc.is_open,
  sacc.is_excluded_from_monitoring,
  h.organisation_supplier_entity_id, 
  h.organisation_entity_id, 
  h.period_start as supplier_period_start,
  h.period_end as supplier_period_end,
  s.type_code as supplier_type_code,
  s.status_code as supplier_status_code,
  s.risk_rating as supplier_risk_rating,
  s.is_supplier_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code as customer_risk_rting_code,
  horg.organisation_id, 
	horg.period_start AS hub_period_start,
	horg.period_end AS hub_period_end,
  sorg.sat_organisation_id,
  sorg.type_code as organisation_type_code,
  sorg.type_concept_id AS organisation_type_concept_id,
  sorg.type_raw_code AS organisation_type_raw_code,
  sorg.type_raw_concept_id AS organisation_type_raw_concept_id,
  sorg.organisation_name,
  sorg.organisation_name_raw,
  sorg.organisation_name_parsed,
  sorg.trading_name,
  sorg.trading_name_raw,
  sorg.alternative_name,
  sorg.alternative_name_raw,
  sorg.duns_number,
  sorg.acronym,
  sorg.organisation_description,
  sorg.organisation_level,
  sorg.parent_organisation_id,
  sorg.primary_adddress_id,
  sorg.legal_structure_type_code,
  sorg.legal_structure_type_concept_id,
  sorg.legal_structure_type_raw_code,
  sorg.legal_structure_type_raw_concept_id,
  sorg.ring_fence_status,
  sorg.company_category_code,
  sorg.company_category_concept_id,
  sorg.company_category_raw_code,
  sorg.company_category_raw_concept_id,
  sorg.company_registration_name,
  sorg.company_registration_number,
  sorg.company_registration_date,
  sorg.company_registration_date_numeric,
  sorg.company_registration_place,
  sorg.company_registration_country_code,
  sorg.company_registration_country_concept_id,
  sorg.company_registration_country_raw_code,
  sorg.company_registration_country_raw_concept_id,
  sorg.primary_operation_country_code,
  sorg.primary_operation_country_concept_id,
  sorg.primary_operation_country_raw_code,
  sorg.primary_operation_country_raw_concept_id,
  sorg.secondary_operation_country_code,
  sorg.secondary_operation_country_concept_id,
  sorg.secondary_operation_country_raw_code,
  sorg.secondary_operation_country_raw_concept_id,
  sorg.company_franchise_type_code,
  sorg.company_franchise_type_concept_id,
  sorg.company_franchise_type_raw_code,
  sorg.company_franchise_type_raw_concept_id,
  sorg.vat_number,
  sorg.vat_registration_date,
  sorg.vat_registration_date_numeric,
  sorg.giin,
  sorg.bic AS organisation_bic,
  sorg.iban AS organisation_iban,
  sorg.legal_entity_identifier,
  sorg.firm_reference_number,
  sorg.unique_tax_payer_reference,
  sorg.capiq,
  sorg.risk_rating_concept_id,
  sorg.risk_rating_raw_code,
  sorg.risk_rating_raw_concept_id,
  sorg.period_start,
	sorg.period_end,
	sorg.load_datetime,
	sorg.record_source_id,
	sorg.data_source_code,
	sorg.data_source_concept_id,
	sorg.data_source_raw_code,
	sorg.data_source_raw_concept_id

    FROM demo_banking_silver.qdp_organisation_suppliers.hub_organisation_suppliers h
    JOIN demo_banking_silver.qdp_organisation_suppliers.sat_organisation_suppliers s
      ON h.organisation_supplier_id = s.organisation_supplier_id
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
      ON horg.organisation_id = s.organisation_id
    JOIN demo_banking_silver.qdp_organisations_all.sat_organisation sorg
      ON horg.organisation_id = sorg.organisation_id
    JOIN demo_banking_silver.qdp_accounts_all.sat_account sacc 
      ON h.organisation_supplier_id = sacc.organisation_customer_id
    JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc 
      ON sacc.account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address saddr
      ON sacc.address_id = saddr.address_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
      ON saddr.address_id = haddr.address_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE sacc.is_account_alarm = true
  
    ORDER BY organisation_supplier_id
;

SELECT * from view_organisation_supplier_bank_accounts_with_alarm;
